In [ ]:
import os
os.environ['HF_TOKEN'] = "YOUR_HF_TOKEN_HERE"
os.environ['NVIDIA_API_KEY'] = "YOUR_NVIDIA_API_KEY_HERE"

API_URL = "https://integrate.api.nvidia.com/v1/chat/completions"
MODEL_NAME = "stepfun-ai/step-3.5-flash"
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/ClinicSpring2026_Results'
OUTPUT_PATH = os.path.join(DRIVE_OUTPUT_DIR, "benchmark_results.json")

MAX_WORKERS = 5 # Reduced from 10 to avoid 429 rate limit errors
REQUEST_TIMEOUT = 120
API_KEY = os.getenv("NVIDIA_API_KEY")


In [ ]:
!pip install -q datasets requests tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import json, time, re, requests, os
from enum import Enum
from typing import List, Dict, Any
from concurrent.futures import ThreadPoolExecutor, as_completed
from datasets import load_dataset
from tqdm.notebook import tqdm

class ReasoningMode(Enum):
    ZERO_SHOT = "zero_shot"
    CHAIN = "chain_of_thought"
    TREE = "tree_of_thought"
    VERIFY = "self_verification"
    FEW_SHOT = "few_shot"

def get_prompt(prompt, mode):
    format_instr = (
        "Keep your reasoning concise and focused. Avoid unnecessary repetition. "
        "Clearly state the final arrangement at the end using ONLY this format:\n"
        "Room 1: [Person], [Pet]\n"
        "Room 2: [Person], [Pet]\n"
        "..."
    )

    legend = (
        "Symbol Legend:\n"
        "- 'X @ Y' means X and Y are in the same room.\n"
        "- 'X <> Y' means X and Y are in DIFFERENT rooms.\n"
        "- 'X: o o # o' refers to the room matching the '#' position (e.g., Room 3).\n"
        "- 'X: o x o o' means X is NOT in the room matching the 'x' position (e.g., Room 2).\n"
        "- '=' and '!=' often refer to adjacency or specific exclusion constraints.\n\n"
    )

    if mode == ReasoningMode.ZERO_SHOT:
        return (
            f"{legend}"
            f"Question:\n{prompt}\n\n"
            f"Please quickly, just give me the answer to this logic puzzle without any explanation. "
            f"{format_instr}"
        )
    elif mode == ReasoningMode.CHAIN:
        return (
            f"{legend}"
            f"Question:\n{prompt}\n\n"
            f"Let's think step by step:\n"
            f"1) List all given constraints and interpret them using the legend.\n"
            f"2) Deduce immediate implications.\n"
            f"3) Explore possibilities and eliminate contradictions.\n"
            f"4) Construct the final solution.\n\n"
            f"{format_instr}"
        )
    elif mode == ReasoningMode.TREE:
        return (
            f"{legend}"
            f"Question:\n{prompt}\n\n"
            f"Solve this puzzle using a Tree-of-Thought approach. Please follow these steps:\n"
            f"1) Tree Search & Branching: Create multiple possible initial assignments based on the first few constraints.\n"
            f"2) Candidate Evaluation: Evaluate each branch against the remaining constraints to see if it is valid. Explain your reasoning for keeping or rejecting each branch.\n"
            f"3) Branch Selection: Prune the invalid branches and expand the valid ones until you find the single branch that satisfies all constraints.\n"
            f"4) Final Conclusion.\n\n"
            f"Consensus Solution:\n"
            f"{format_instr}"
        )
    elif mode == ReasoningMode.VERIFY:
        return (
            f"{legend}"
            f"Question:\n{prompt}\n\n"
            f"First, solve the puzzle using the symbol legend.\n"
            f"Then, verify your solution against every constraint in the question.\n"
            f"Final Verified Solution:\n"
            f"{format_instr}"
        )
    elif mode == ReasoningMode.FEW_SHOT:
        example = (
            "Example Puzzle:\n"
            "There are 3 rooms assigned to Alice, Bob, Charlie with pets: anole, bat, cat.\n"
            "1. Bob @ anole\n2. Alice <> bat\n"
            "Solution:\n"
            "Room 1: Bob, anole\nRoom 2: Alice, cat\nRoom 3: Charlie, bat\n\n"
        )
        return (
            f"{legend}"
            f"{example}"
            f"Question:\n{prompt}\n\n"
            f"{format_instr}"
        )
    return prompt

def call_api(prompt, max_retries=3):
    headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.3,
        "max_tokens": 1024
    }
    
    for attempt in range(max_retries):
        try:
            r = requests.post(API_URL, json=payload, headers=headers, timeout=REQUEST_TIMEOUT)
            
            # Retry logic for Rate Limiting (429)
            if r.status_code == 429:
                wait_time = (attempt + 1) * 3
                time.sleep(wait_time)
                continue
            
            if r.status_code != 200: return {"error": f"HTTP {r.status_code}: {r.text}"}
            data = r.json()
            if 'choices' not in data or not data['choices']:
                return {"error": f"Unexpected API response: {json.dumps(data)}"}
            
            message = data["choices"][0]["message"]
            content = message.get("content") or ""
            reasoning = message.get("reasoning_content") or message.get("reasoning") or ""
            
            full_resp = f"--- Reasoning ---\n{reasoning}\n\n{content}" if reasoning else content
            return {"response": full_resp, "latency": r.elapsed.total_seconds()}
        except Exception as e:
            if attempt == max_retries - 1: return {"error": f"Client Error: {str(e)}"}
            time.sleep(2)
    return {"error": "Max retries exceeded for 429 Rate Limit"}

def extract_words(text):
    if text is None: return set()
    return set(re.findall(r"\b\w+\b", str(text).lower()))

def grade(pred, gt):
    if pred is None or gt is None:
        return 0.0
    p, g = extract_words(pred), extract_words(gt)
    return len(p & g) / max(len(g), 1)

def save_results(results):
    dir_name = os.path.dirname(OUTPUT_PATH)
    if dir_name: os.makedirs(dir_name, exist_ok=True)
    with open(OUTPUT_PATH, "w") as f: json.dump(results, f, indent=2)


In [ ]:
def run(limit=1000):
    print("Loading dataset...")
    ds = load_dataset("emunah/deductive_logical_reasoning-room_assignment", split="train", token=os.getenv("HF_TOKEN"))
    ans_k = "completion" if "completion" in ds.column_names else "answer"
    ds = ds.shuffle(seed=42).select(range(min(limit, len(ds))))
    results = []

    for mode in ReasoningMode:
        print(f"\n=== Running {mode.name} ===")
        def process(i):
            item = ds[i]
            res = call_api(get_prompt(item["question"], mode))
            if "error" in res: return {"index": i, "mode": mode.name, "error": res["error"]}
            return {
                "index": i, "mode": mode.name, "response": res["response"], 
                "ground_truth": item[ans_k], "score": grade(res["response"], item[ans_k]), "latency": res["latency"]
            }

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = [ex.submit(process, i) for i in range(len(ds))]
            for f in tqdm(as_completed(futures), total=len(futures)):
                results.append(f.result())
                if len(results) % 20 == 0: save_results(results)
    
    save_results(results)
    print("\n=== SUMMARY ===")
    summary = {}
    for r in results:
        if "error" not in r: summary.setdefault(r["mode"], []).append(r["score"])
    for mode, scores in summary.items():
        if scores: print(f"{mode}: {sum(scores)/len(scores):.3f}")

run(limit=1000)
